In [1]:
#Importo pandas para abrir el df

import pandas as pd

In [2]:
# Carga de datos desde Google Drive
FILE_ID = '1xtfIVuyOoayCfLOyAOe4jfZ2wOwpPs9R'
url = f'https://drive.google.com/uc?id={FILE_ID}'

# Leemos el archivo directamente a un DataFrame de Pandas
try:
    df = pd.read_csv(url)
    print(f"Dataset cargado con éxito. Dimensiones: {df.shape[0]} filas y {df.shape[1]} columnas.")
    display(df.head())
except Exception as e:
    print(f"Error al cargar el archivo: {e}. Verifica el ID o los permisos de Drive.")

Dataset cargado con éxito. Dimensiones: 6045 filas y 6 columnas.


,RIAGENDR,RIDAGEYR,BMXWT,BMXHT,MGX_MAXHT,DXD_ASM_BMI_LOW
0,1,22,69.2,172.3,34.9,0
1,2,44,67.2,170.1,38.3,0
2,1,21,54.6,164.8,47.4,0
3,2,43,98.6,172.0,35.5,0
4,1,55,91.4,182.1,60.6,0


In [3]:
#Veo el dataset

df

,RIAGENDR,RIDAGEYR,BMXWT,BMXHT,MGX_MAXHT,DXD_ASM_BMI_LOW
0,1,22,69.2,172.3,34.9,0
1,2,44,67.2,170.1,38.3,0
2,1,21,54.6,164.8,47.4,0
3,2,43,98.6,172.0,35.5,0
4,1,55,91.4,182.1,60.6,0
...,...,...,...,...,...,...
6040,1,58,76.3,170.4,43.9,0
6041,1,52,79.5,176.2,47.9,0
6042,1,40,79.0,171.7,46.4,0
6043,1,26,71.8,171.3,37.2,0


In [4]:
#Hay missing values?

missing_values = df.isnull().sum()
print("Missing Values per column:\n", missing_values)


Missing Values per column:
 RIAGENDR           0
RIDAGEYR           0
BMXWT              0
BMXHT              0
MGX_MAXHT          0
DXD_ASM_BMI_LOW    0
dtype: int64


In [6]:
#¿Cuantos valores válidos tengo en cada variable?

valid_counts = df.count()
print("\nNumber of Valid Values per column:\n", valid_counts)


Number of Valid Values per column:
 RIAGENDR           6045
RIDAGEYR           6045
BMXWT              6045
BMXHT              6045
MGX_MAXHT          6045
DXD_ASM_BMI_LOW    6045
dtype: int64


In [10]:
# Defino mis variables RIAGENDR y DXD_ASM_BMI_LOW son categoricas. Las otras son continuas.

categorical_vars = ['RIAGENDR', 'DXD_ASM_BMI_LOW']
for col in categorical_vars:
    if col in df.columns:
        df[col] = pd.Categorical(df[col])
    else:
        print(f"Warning: Column '{col}' not found in the DataFrame.")


In [11]:
#Que tipo de variables tengo?

print("\nData Types of each column:\n", df.dtypes)


Data Types of each column:
 RIAGENDR           category
RIDAGEYR              int64
BMXWT               float64
BMXHT               float64
MGX_MAXHT           float64
DXD_ASM_BMI_LOW    category
dtype: object


In [12]:
#Importo las librerias necesarias
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Defino covariables(X) y variable objetivo (Y)
X = df[['RIAGENDR', 'RIDAGEYR', 'BMXWT', 'BMXHT', 'MGX_MAXHT']]
Y = df['DXD_ASM_BMI_LOW']

#Divido en entrenamiento y testeo (80% - 20%)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

#Defino mis hiperparametros (pruning-poda), quiero evitar el sobreajuste
param_grid = {
    'max_depth': [2, 3, 4, 5],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

#Defino el modelo
dtc = DecisionTreeClassifier(random_state=42)

#GridSearchCV para validación cruzada
grid_search = GridSearchCV(estimator=dtc, param_grid=param_grid, cv=5, scoring='f1')

#Entreno a mi modelo
grid_search.fit(X_train, Y_train)


#Selecciono el mejor modelo
clf = grid_search.best_estimator_


#Hago mis predicciones en el grupo de testeo
Y_pred = clf.predict(X_test)

#Calculo las métricas
accuracy = accuracy_score(Y_test, Y_pred)
precision = precision_score(Y_test, Y_pred, zero_division=0)
recall = recall_score(Y_test, Y_pred, zero_division=0)
f1 = f1_score(Y_test, Y_pred, zero_division=0)
roc_auc = roc_auc_score(Y_test, clf.predict_proba(X_test)[:, 1])
conf_matrix = confusion_matrix(Y_test, Y_pred)

#Las imprimo
print(f"Tasa de aciertos: {accuracy}")
print(f"Precision: {precision}")
print(f"Sensibilidad: {recall}")
print(f"F1 Score: {f1}")
print(f"ROC AUC Score: {roc_auc}")
print(f"Confusion Matrix:\n{conf_matrix}")

Tasa de aciertos: 0.9396195202646815
Precision: 0.6222222222222222
Sensibilidad: 0.3333333333333333
F1 Score: 0.43410852713178294
ROC AUC Score: 0.8529788359788361
Confusion Matrix:
[[1108   17]
 [  56   28]]


In [13]:
from sklearn import tree

#Veo el arbol entrenado
plt.figure(figsize=(150,75)) # Increased figure size for better readability
tree.plot_tree(clf, filled=True, feature_names=X.columns.tolist(), class_names=[str(c) for c in Y.unique()])
plt.show()

In [14]:
#Quedó un poco grande no? Lindo arbol para sombra
#Que pasó?

#Hago las predicciones en el grupo de entrenamiento
Y_train_pred = clf.predict(X_train)
Y_train_proba = clf.predict_proba(X_train)[:, 1]

#Calculo las métricas en el grupo de entrenamiento
train_accuracy = accuracy_score(Y_train, Y_train_pred)
train_precision = precision_score(Y_train, Y_train_pred, zero_division=0)
train_recall = recall_score(Y_train, Y_train_pred, zero_division=0)
train_f1 = f1_score(Y_train, Y_train_pred, zero_division=0)
train_roc_auc = roc_auc_score(Y_train, Y_train_proba)
train_conf_matrix = confusion_matrix(Y_train, Y_train_pred)

#Imprimo las métricas
print("Métricas en el set de entrenamiento:")
print(f"Tasa de aciertos: {train_accuracy}")
print(f"Precision: {train_precision}")
print(f"Sensibilidad: {train_recall}")
print(f"F1 Score: {train_f1}")
print(f"ROC AUC Score: {train_roc_auc}")
print(f"Confusion Matrix:\n{train_conf_matrix}")

Métricas en el set de entrenamiento:
Tasa de aciertos: 0.946236559139785
Precision: 0.6814159292035398
Sensibilidad: 0.4502923976608187
F1 Score: 0.5422535211267606
ROC AUC Score: 0.9040120420469657
Confusion Matrix:
[[4422   72]
 [ 188  154]]
